# Flagged-site review and Sentinel availability

Inspect public current OSM outlines and identity evidence, then list CDSE Sentinel-1 GRD and Sentinel-2 L2A catalogue footprints for March-April 2022. No imagery pixels or Earth Engine assets are queried. Start after notebooks 00, 01 and 03 have produced their registered inputs.

Raw inputs are retained; fresh downloads need internet access. Site identity proposals are documented separately from the original inventory. Reruns replace outputs and record a new run. The reference-date windows and candidate track are exploratory, not validated flood periods or models.

In [ ]:
from pathlib import Path
import json

# In Colab, clone the repository and change into it before running this cell.
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "config/ethekwini.yaml").is_file()), None)
if ROOT is None:
    raise RuntimeError("Run from the africa-map checkout or one of its subdirectories.")
CONFIG = json.loads((ROOT / "config/ethekwini.yaml").read_text())
print("Project:", CONFIG["project_id"])


In [ ]:
from urllib.request import urlopen, Request
from urllib.parse import urlencode, urljoin
from datetime import datetime, timezone
import ssl
import certifi

SAT = ROOT / CONFIG["availability"]["source_directory"]
SITE = ROOT / CONFIG["site_review"]["source_directory"]
SAT.mkdir(parents=True, exist_ok=True)
SITE.mkdir(parents=True, exist_ok=True)
# Retain existing raw snapshots. On a fresh checkout this cell downloads public
# metadata and identity-reference documents only, never satellite image pixels.
for directory, requests in [(SITE, {
    **{f"way_{i}_full.json": f"https://api.openstreetmap.org/api/0.6/way/{i}/full.json" for i in CONFIG["site_review"]["way_ids"]},
    "library_surroundings.json": "https://api.openstreetmap.org/api/0.6/map.json?bbox=30.8739,-30.0590,30.8780,-30.0560",
    "liasa_kzn_libraries.pdf": "https://cdn.ymaws.com/www.liasa.org.za/resource/resmgr/documents_/other/libraries_add_book.pdf",
    "municipal_umkhumbane_project.pdf": "https://dag.durban.gov.za/uploads/0000/6/2025/10/02/umkhambane-museum-2014-2016.pdf",
})]:
    manifest_path = directory / "download_manifest.json"
    manifest = json.loads(manifest_path.read_text()) if manifest_path.exists() else []
    for filename, url in requests.items():
        if (directory / filename).exists():
            continue
        context = ssl.create_default_context(cafile=certifi.where())
        if "durban.gov.za" in url:
            with urlopen("http://crt.sectigo.com/SectigoPublicServerAuthenticationCADVR36.crt", timeout=30) as response:
                context.load_verify_locations(cadata=ssl.DER_cert_to_PEM_cert(response.read()))
        with urlopen(Request(url, headers={"User-Agent": "AfricaHeritageFloodResearch/0.1"}), context=context, timeout=60) as response:
            content = response.read()
        if filename.endswith(".pdf"):
            assert content.startswith(b"%PDF")
        else:
            json.loads(content)
        (directory / filename).write_bytes(content)
        manifest.append({"file": filename, "url": url, "accessed_at_utc": datetime.now(timezone.utc).isoformat()})
    manifest_path.write_text(json.dumps(manifest, indent=2) + "\n")

manifest_path = SAT / "download_manifest.json"
manifest = json.loads(manifest_path.read_text()) if manifest_path.exists() else []
for collection in ["sentinel-1-grd", "sentinel-2-l2a"]:
    requests = {collection + "_collection.json": "https://stac.dataspace.copernicus.eu/v1/collections/" + collection}
    bbox = "30.55999994,-30.26878394,31.18619007,-29.50113007"
    url = "https://stac.dataspace.copernicus.eu/v1/search?" + urlencode({
        "collections": collection, "bbox": bbox,
        "datetime": CONFIG["availability"]["search_start"] + "/" + CONFIG["availability"]["search_end_exclusive"], "limit": 100})
    page = 0
    while url:
        name = f"audit_{collection}_page_{page:03d}.json"
        if (SAT / name).exists():
            payload = json.loads((SAT / name).read_text())
        else:
            with urlopen(url, timeout=60) as response:
                content = response.read()
            payload = json.loads(content)
            if "features" not in payload:
                raise RuntimeError("Invalid STAC response")
            (SAT / name).write_bytes(content)
            manifest.append({"file": name, "url": url, "accessed_at_utc": datetime.now(timezone.utc).isoformat()})
        nxt = next((link for link in payload.get("links", []) if link["rel"] == "next"), None)
        if nxt and nxt.get("method", "GET") != "GET":
            raise RuntimeError("Review non-GET pagination before continuing")
        url = urljoin(url, nxt["href"]) if nxt else None
        page += 1
        if page > 100:
            raise RuntimeError("Unexpected pagination length")
    for name, url in requests.items():
        if not (SAT / name).exists():
            with urlopen(url, timeout=60) as response:
                content = response.read()
            json.loads(content); (SAT / name).write_bytes(content)
            manifest.append({"file": name, "url": url, "accessed_at_utc": datetime.now(timezone.utc).isoformat()})
# Independent public catalogue protocol check, still the same CDSE archive.
# It is not an independent sensor or flood-validation source.
flt = "Collection/Name eq 'SENTINEL-1' and ContentDate/Start ge 2022-03-01T00:00:00.000Z and ContentDate/Start lt 2022-05-01T00:00:00.000Z and OData.CSC.Intersects(area=geography'SRID=4326;POLYGON((30.56 -30.269,31.187 -30.269,31.187 -29.501,30.56 -29.501,30.56 -30.269))') and Attributes/OData.CSC.StringAttribute/any(att:att/Name eq 'productType' and att/OData.CSC.StringAttribute/Value eq 'IW_GRDH_1S')"
url = "https://catalogue.dataspace.copernicus.eu/odata/v1/Products?" + urlencode({"$filter": flt, "$top": 100, "$expand": "Attributes", "$count": "true"})
page = 0
while url:
    name = f"sentinel-1-odata_page_{page:03d}.json"
    if (SAT / name).exists():
        payload = json.loads((SAT / name).read_text())
    else:
        with urlopen(url, timeout=60) as response:
            content = response.read()
        payload = json.loads(content); (SAT / name).write_bytes(content)
        manifest.append({"file": name, "url": url, "accessed_at_utc": datetime.now(timezone.utc).isoformat()})
    url = payload.get("@odata.nextLink"); page += 1
manifest_path.write_text(json.dumps(manifest, indent=2) + "\n")
print("Public metadata and site evidence available; no image pixels downloaded.")


In [ ]:
import csv
import hashlib
import platform
import subprocess
from collections import Counter
import geopandas as gpd
import pandas as pd
import shapely
import pyproj
from heritage_flood.sources import osm_way_outline, scene_manifest

started = datetime.now(timezone.utc)
run_id = "site_availability_" + started.strftime("%Y%m%dT%H%M%S%fZ")
output = ROOT / CONFIG["output_directory"]
points = gpd.read_file(output / "institutions.geojson")
boundary = gpd.read_file(ROOT / CONFIG["boundary_path"])
# Ensure retained boundary, institution and floodplain bytes match registered runs.
with (ROOT / "metadata/datasets.csv").open(newline="") as handle:
    datasets = {r["dataset_id"]: r for r in csv.DictReader(handle)}
assert hashlib.sha256((ROOT / CONFIG["boundary_path"]).read_bytes()).hexdigest() == datasets[CONFIG["boundary_dataset_id"]]["sha256"]
with (ROOT / "metadata/processing_runs.csv").open(newline="") as handle:
    runs = list(csv.DictReader(handle))
for path, prefix in [(output / "institutions.geojson", "inventory_audit_"), (output / "floodplain_screening.gpkg", "floodplain_screen_")]:
    recorded = next(r for r in reversed(runs) if r["run_id"].startswith(prefix))
    hashes = json.loads((ROOT / recorded["output_checksums_path"]).read_text())
    assert hashlib.sha256(path.read_bytes()).hexdigest() == hashes[str(path.relative_to(ROOT))]
flood = gpd.read_file(output / "floodplain_screening.gpkg", layer="mapped_floodplain").to_crs(CONFIG["analysis_crs"]).geometry.union_all()
footprints = []
for way_id in CONFIG["site_review"]["way_ids"]:
    payload = json.loads((SITE / f"way_{way_id}_full.json").read_text())
    way = next(e for e in payload["elements"] if e["type"] == "way" and e["id"] == way_id)
    polygon = osm_way_outline(payload, way_id)
    metric_polygon = gpd.GeoSeries([polygon], crs=4326).to_crs(CONFIG["analysis_crs"]).iloc[0]
    source_point = points.loc[points.institution_id == f"osm:way/{way_id}"].geometry.iloc[0]
    overlap_area = metric_polygon.intersection(flood).area
    footprints.append({"institution_id": f"osm:way/{way_id}", "osm_way_version": way["version"],
        "osm_way_edited_at": way["timestamp"],
        "latest_member_node_edit_at": max(e["timestamp"] for e in payload["elements"] if e["type"] == "node"),
        "source_point_within_outline": polygon.covers(source_point),
        "outline_area_m2": metric_polygon.area, "mapped_overlap_area_m2": overlap_area,
        "mapped_overlap_fraction": min(1.0, overlap_area / metric_polygon.area),
        "distance_outline_to_mapped_floodplain_m": metric_polygon.distance(flood),
        "geometry_review_status": "current_osm_outline_only_not_ground_verified",
        "geometry": polygon})
footprints = gpd.GeoDataFrame(footprints, geometry="geometry", crs=4326)
assert footprints.source_point_within_outline.all()
footprints.to_crs(CONFIG["analysis_crs"]).to_file(output / "flagged_site_review.gpkg", layer="osm_outlines", driver="GPKG")
points.loc[points.institution_id.isin(footprints.institution_id)].to_crs(CONFIG["analysis_crs"]).to_file(output / "flagged_site_review.gpkg", layer="source_points", driver="GPKG")
footprints.drop(columns="geometry").to_csv(output / "flagged_site_geometry_review.csv", index=False)
# These are evidence-qualified review proposals, not edits to OSM or the inventory.
decisions = [
 {"institution_id": "osm:way/368501740", "source_name": "", "proposed_name": "Amanzimtoti Library",
  "identity_review_status": "candidate_address_consistent", "evidence": "LIASA directory printed p54/PDF p62: Civic Centre, 1-18 Riverside Road; OSM surroundings include Riverside and Hutchison Roads",
  "limitations": "Directory PDF creation metadata dates to 2013; exact building identity and current operation not independently confirmed; source name remains blank"},
 {"institution_id": "osm:way/1199992499", "source_name": "Umkhumbane Museum", "proposed_name": "Umkhumbane Museum",
  "identity_review_status": "name_and_locality_corroborated", "evidence": "Municipal project sheet: museum at 2 Kalanden Road, Cato Manor, contract period June 2014-April 2016",
  "limitations": "Document supports identity/locality, not exact outline or April 2022 flood damage; current OSM geometry postdates event"},
]
for row in decisions:row["review_run_id"] = run_id
pd.DataFrame(decisions).to_csv(ROOT / "metadata/site_review_decisions.csv", index=False)
print(footprints.drop(columns="geometry").to_string(index=False))


In [ ]:
items = []
for collection in ["sentinel-1-grd", "sentinel-2-l2a"]:
    page = 0
    while True:
        path = SAT / f"audit_{collection}_page_{page:03d}.json"
        payload = json.loads(path.read_text())
        items.extend(payload["features"])
        if not any(link["rel"] == "next" for link in payload.get("links", [])):
            break
        page += 1
search = CONFIG["availability"]
items = [item for item in items if pd.Timestamp(search["search_start"]) <= pd.Timestamp(item["properties"]["datetime"]) < pd.Timestamp(search["search_end_exclusive"])]
scenes = scene_manifest(items, boundary, points, analysis_crs=CONFIG["analysis_crs"])
scenes["acquisition_date_utc"] = pd.to_datetime(scenes.acquired_at_utc, utc=True).dt.strftime("%Y-%m-%d")
scenes["run_id"] = run_id
scenes.to_file(output / "sentinel_scene_footprints.gpkg", layer="catalogue_footprints", driver="GPKG")
scenes.drop(columns="geometry").sort_values(["catalogue_collection", "acquired_at_utc", "catalogue_id"]).to_csv(output / "sentinel_scene_manifest.csv", index=False)
# Catalogue protocol cross-check at acquisition identity level: COG/SAFE
# processing versions have different suffixes and are not interchangeable assets.
odata = []
page = 0
while True:
    payload = json.loads((SAT / f"sentinel-1-odata_page_{page:03d}.json").read_text())
    odata.extend(payload["value"])
    if not payload.get("@odata.nextLink"):
        break
    page += 1
expected_odata_count = json.loads((SAT / "sentinel-1-odata_page_000.json").read_text())["@odata.count"]
assert len(odata) == expected_odata_count
stac_keys = {"_".join(item["id"].split("_")[:7]) for item in items if item["collection"] == "sentinel-1-grd"}
odata_keys = {"_".join(item["Name"].split("_")[:7]) for item in odata}
assert stac_keys == odata_keys
# Catalogue candidates only. Pixel masks/terrain effects must be checked later.
review_ids = list(footprints.institution_id)
candidate_rows = []
for collection in ["sentinel-1-grd", "sentinel-2-l2a"]:
    subset = scenes.loc[scenes.catalogue_collection == collection]
    for row in subset.itertuples():
        covered = set(row.covered_institution_ids.split(";"))
        if not set(review_ids) <= covered:
            continue
        if collection == "sentinel-1-grd" and row.relative_orbit == 43 and row.orbit_state == "ascending" and row.acquisition_date_utc in ["2022-04-05", "2022-04-17"]:
            role = "candidate_sar_baseline" if row.acquisition_date_utc == "2022-04-05" else "candidate_sar_post_event"
        elif collection == "sentinel-2-l2a" and row.grid_code == "MGRS-36JTM" and row.acquisition_date_utc in ["2022-03-29", "2022-04-13", "2022-04-28"]:
            role = {"2022-03-29": "candidate_optical_baseline", "2022-04-13": "early_post_event_high_scene_cloud", "2022-04-28": "late_post_event_candidate"}[row.acquisition_date_utc]
        else:
            continue
        candidate_rows.append({"role": role, "catalogue_id": row.catalogue_id, "acquired_at_utc": row.acquired_at_utc,
            "relative_orbit": row.relative_orbit, "orbit_state": row.orbit_state, "sar_mode": row.sar_mode,
            "polarizations": row.polarizations, "grid_code": row.grid_code, "cloud_cover_scene_pct": row.cloud_cover_scene_pct,
            "pixel_usability": "not_evaluated", "earth_engine_asset_id": "", "run_id": run_id})
candidates = pd.DataFrame(candidate_rows)
candidates.to_csv(output / "sentinel_candidate_scenes.csv", index=False)
summary = {
    "run_id": run_id, "catalogue": "Copernicus Data Space public STAC; Sentinel-1 acquisition keys cross-checked through CDSE OData",
    "search": search, "scene_counts": dict(Counter(scenes.catalogue_collection)),
    "sar_odata_acquisition_keys_match": True,
    "acquisition_dates_by_collection": {c: sorted(set(g.acquisition_date_utc)) for c, g in scenes.groupby("catalogue_collection")},
    "candidate_scene_count": len(candidates), "earth_engine_authenticated": None,
    "earth_engine_authentication_checked_in_notebook": False,
    "earth_engine_status": "Initial local ee.Initialize check on 2026-10-20 required authorization; this notebook does not recheck authentication or query Earth Engine",
    "pixel_data_downloaded": False, "pixel_usability": "not_evaluated",
    "feasibility": "Conditional for residual-water/visible-impact exploration; peak flood extent cannot be reconstructed or validated from these metadata alone",
    "temporal_limit": "Candidate matched SAR acquisitions 5 and 17 April bracket the reported 11-12 April peak; a dry 17 April observation would not rule out earlier inundation",
    "cloud_limit": "Sentinel-2 scene-level cloud percentages are not site-level clear-pixel measurements",
}
(ROOT / "metadata/ethekwini_availability_summary.json").write_text(json.dumps(summary, indent=2) + "\n")
print(candidates.to_string(index=False))


In [ ]:
run_directory = ROOT / "metadata/runs" / run_id
run_directory.mkdir(parents=True, exist_ok=False)
(run_directory / "config.json").write_text(json.dumps(CONFIG, indent=2) + "\n")
env = {"python": platform.python_version(), "geopandas": gpd.__version__, "pandas": pd.__version__, "shapely": shapely.__version__, "pyproj": pyproj.__version__}
(run_directory / "environment.json").write_text(json.dumps(env, indent=2) + "\n")
site_files = [SITE / f"way_{i}_full.json" for i in CONFIG["site_review"]["way_ids"]] + [SITE / n for n in ["library_surroundings.json", "liasa_kzn_libraries.pdf", "municipal_umkhumbane_project.pdf"]]
satellite_files = sorted(SAT.glob("audit_*_page_*.json")) + sorted(SAT.glob("sentinel-1-odata_page_*.json")) + [SAT / (c + "_collection.json") for c in ["sentinel-1-grd", "sentinel-2-l2a"]]
inputs = site_files + satellite_files + [ROOT / CONFIG["boundary_path"], output / "institutions.geojson", output / "floodplain_screening.gpkg"]
(run_directory / "input_checksums.json").write_text(json.dumps({str(p.relative_to(ROOT)): hashlib.sha256(p.read_bytes()).hexdigest() for p in inputs}, indent=2) + "\n")
code = [ROOT / "src/heritage_flood/sources.py", ROOT / "notebooks/02_data_availability.ipynb"]
(run_directory / "code_checksums.json").write_text(json.dumps({str(p.relative_to(ROOT)): hashlib.sha256(p.read_bytes()).hexdigest() for p in code}, indent=2) + "\n")
source_rows = []
groups = [
    ("osm_flagged_sites_snapshot_20261020", "Current OSM flagged-site outlines and surroundings", site_files[:3], "OpenStreetMap contributors", "https://www.openstreetmap.org/copyright", "ODbL 1.0; preserve attribution and applicable database obligations", "Current OSM, not ground truth or verified April 2022 geometry"),
    ("site_identity_documents_20261020", "Public heritage/library identity evidence", site_files[3:], "LIASA-hosted library directory; eThekwini municipal project sheet", "research/site_review_and_availability.md", "No explicit open redistribution license established; retained locally", "Library directory PDF creation metadata dates to 2013; museum sheet describes 2014-2016 project; neither proves 2022 flood damage"),
    ("cdse_sentinel_catalogue_snapshot_20261020", "Sentinel catalogue metadata March-April 2022", satellite_files, "Copernicus Data Space Ecosystem", "https://stac.dataspace.copernicus.eu/v1", "Collections declare other and link Copernicus Sentinel legal notices; consult retained collection metadata", "Catalogue footprints/scene cloud statistics only; no pixels inspected; CDSE IDs are not Earth Engine asset IDs"),
]
source_manifest_paths = []
for dataset_id, title, files, provider, url, license_text, limitations in groups:
    path = ROOT / "metadata" / (dataset_id + "_checksums.json")
    path.write_text(json.dumps({str(p.relative_to(ROOT)): hashlib.sha256(p.read_bytes()).hexdigest() for p in files}, indent=2) + "\n")
    source_manifest_paths.append(path)
    source_rows.append({"dataset_id": dataset_id, "title": title, "provider": provider, "version": "Snapshot inspected 2026-10-20",
        "source_url_or_asset_id": url, "status": "accessed", "accessed_at_utc": started.isoformat(),
        "observation_start": "2022-03-01" if dataset_id.startswith("cdse") else "",
        "observation_end": "2022-04-30" if dataset_id.startswith("cdse") else "",
        "geographic_coverage": "eThekwini candidate sites / municipality", "crs": "EPSG:4326 geometry where present",
        "license": license_text, "redistribution_conditions": "Raw documents and spatial artifacts remain local; inspect source-specific terms before publication",
        "limitations": limitations + "; register sha256 hashes an input-file checksum manifest", "storage_location": str(path.relative_to(ROOT)), "sha256": hashlib.sha256(path.read_bytes()).hexdigest()})
outputs = [output / name for name in ["flagged_site_review.gpkg", "flagged_site_geometry_review.csv", "sentinel_scene_footprints.gpkg", "sentinel_scene_manifest.csv", "sentinel_candidate_scenes.csv"]]
outputs += [ROOT / "metadata/site_review_decisions.csv", ROOT / "metadata/ethekwini_availability_summary.json"] + source_manifest_paths
(run_directory / "checksums.json").write_text(json.dumps({str(p.relative_to(ROOT)): hashlib.sha256(p.read_bytes()).hexdigest() for p in outputs}, indent=2) + "\n")
run_row = {"run_id": run_id, "started_at_utc": started.isoformat(),
    "input_dataset_ids": ";".join([g[0] for g in groups] + [CONFIG["institution_dataset_id"], CONFIG["boundary_dataset_id"], CONFIG["floodplain_dataset_id"]]),
    "code_commit": subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=ROOT, text=True).strip(),
    "working_tree_dirty": str(bool(subprocess.check_output(["git", "status", "--porcelain"], cwd=ROOT, text=True).strip())).lower(),
    "config_snapshot_path": str((run_directory / "config.json").relative_to(ROOT)), "environment_path": str((run_directory / "environment.json").relative_to(ROOT)),
    "method": "Inspect current OSM way/node geometry; public document identity proposals; complete STAC metadata query and OData SAR acquisition cross-check; no pixel analysis",
    "scene_manifest_path": str((output / "sentinel_scene_manifest.csv").relative_to(ROOT)),
    "output_locations": ";".join(str(p.relative_to(ROOT)) for p in outputs), "output_checksums_path": str((run_directory / "checksums.json").relative_to(ROOT)),
    "validation_status": "partial", "notes": "Metadata geometry/provenance checks passed; pixel usability, peak inundation and exact building identity remain unverified. Earth Engine requires user authentication. See research/site_review_and_availability.md."}
for filename, key, new_rows in [("datasets.csv", "dataset_id", source_rows), ("processing_runs.csv", "run_id", [run_row])]:
    path = ROOT / "metadata" / filename
    with path.open(newline="") as handle:
        reader = csv.DictReader(handle); fields, rows = reader.fieldnames, list(reader)
    for row in new_rows:
        previous = next((r for r in rows if r[key] == row[key]), None)
        if filename == "datasets.csv" and previous and previous["sha256"] != row["sha256"]:
            raise ValueError("Source snapshot changed: use a new dataset ID")
        rows = [r for r in rows if r[key] != row[key]] + [row]
    with path.open("w", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=fields, lineterminator="\n"); writer.writeheader(); writer.writerows(rows)
print("Recorded metadata/site review run:", run_id)


## Interpretation and next action

Read `research/site_review_and_availability.md`. Current OSM outlines and a historical municipal floodplain are temporally mismatched and not ground truth. Site-level optical cloud masks, SAR artifacts and flood signatures remain unchecked. SAR April 5/17 is a candidate comparison that misses the reported peak; absence of water on April 17 would not rule out earlier flooding.

Earth Engine did not initialize during this audit because authorization is required. For pixel analysis, run `earthengine authenticate` interactively, provide an authorized Cloud project ID in configuration, and initialize with that project. Do not commit credentials or notebook authentication outputs. Match catalogue candidates to actual Earth Engine image timestamps/orbits and inspect masks before choosing analysis windows. No image exports should be launched until that check.